# Cow breed classifier v2: transfer learning, no data leakage, `.keras` format

**Run this on Google Colab with a GPU** (Runtime → Change runtime type → T4 GPU).

What changed from v1 (`CowBreedClassification.ipynb`) and why:

| v1 problem | v2 fix |
|---|---|
| Augmented copies were concatenated **then** split → variants of training photos landed in val/test | Split the *files* first (stratified 70/15/15); augmentation runs only inside the model, only while training |
| `image_dataset_from_directory` reshuffles every epoch, so `take`/`skip` gave different splits each epoch | Fixed, seeded split saved to `splits.json` |
| Trained on pixels ÷255 but `main.py` sends raw 0–255 → production predictions were off | Model takes **raw 0–255 RGB** (EfficientNet rescales internally), exactly what the API sends |
| Small CNN from scratch on ~2k images (~53% val accuracy) | EfficientNetB0 pre-trained on ImageNet, then careful fine-tuning |
| Vertical flips (upside-down cows don't exist) | Realistic augmentation: horizontal flip, small rotation/zoom/shift, lighting |
| No early stopping | EarlyStopping + ReduceLROnPlateau, best weights restored |
| Saved as legacy `.h5` | Saved as `.keras` + `model_meta.json` (class order, image size, versions) |

**How we check for overfitting:** evaluate train, validation and test the *same way* (no augmentation, no dropout) and compare. A big train–val gap = memorising, not learning.

## 1. Setup

In [5]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, hashlib, random
import numpy as np
import tensorflow as tf
import keras
from keras import layers
import matplotlib.pyplot as plt

print('TensorFlow', tf.__version__, '| Keras', keras.__version__)
print('GPU:', tf.config.list_physical_devices('GPU') or 'NONE: Runtime > Change runtime type > T4 GPU')

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TensorFlow 2.21.0 | Keras 3.13.2
GPU: NONE: Runtime > Change runtime type > T4 GPU


In [6]:
DATA_DIR = '/content/drive/My Drive/dataset'    # one sub-folder per breed (later: add a 'not_cow' folder)
OUT_DIR  = '/content/drive/My Drive/cow_model'  # trained model + metadata are saved here
IMG_SIZE = 224                                   # EfficientNetB0's native size
BATCH    = 32
os.makedirs(OUT_DIR, exist_ok=True)

## 2. Scan the dataset: drop unreadable files and exact duplicates

Scraped datasets often contain the same photo twice. If one copy lands in train and the other in test,
the test score is inflated. We hash every file and keep only the first copy.

In [7]:
EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.gif')
class_names = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))
print('Classes:', class_names)

paths, labels, seen, broken, dupes = [], [], {}, [], []
for idx, name in enumerate(class_names):
    folder = os.path.join(DATA_DIR, name)
    for f in sorted(os.listdir(folder)):
        if not f.lower().endswith(EXTS):
            continue
        p = os.path.join(folder, f)
        with open(p, 'rb') as fh:
            raw = fh.read()
        digest = hashlib.md5(raw).hexdigest()
        if digest in seen:
            dupes.append((p, seen[digest]))
            continue
        try:
            tf.io.decode_image(raw, channels=3, expand_animations=False)
        except Exception:
            broken.append(p)
            continue
        seen[digest] = p
        paths.append(p)
        labels.append(idx)

print(f'Usable images: {len(paths)} | duplicates skipped: {len(dupes)} | unreadable skipped: {len(broken)}')
cross_class = [(a, b) for a, b in dupes if os.path.dirname(a) != os.path.dirname(b)]
if cross_class:
    print(f'WARNING: {len(cross_class)} identical images are filed under DIFFERENT breeds, e.g.:')
    for a, b in cross_class[:5]:
        print('  ', a, '<->', b)

Classes: ['Ayrshire', 'Black-Angus', 'Boran', 'Guernsey', 'Holstein', 'Jersey', 'Sahiwal', 'Zebu']


KeyboardInterrupt: 

In [ ]:
counts = np.bincount(labels, minlength=len(class_names))
plt.figure(figsize=(9, 3.5))
plt.bar(class_names, counts, color='#1b5e37')
for i, c in enumerate(counts):
    plt.text(i, c, str(c), ha='center', va='bottom', fontsize=9)
plt.title('Images per breed'); plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()
print('Smallest class has', counts.min(), 'images. Aim for 200+ per class; under ~100 the model will struggle on that breed.')

## 3. Split the files once (stratified 70 / 15 / 15)

Every breed keeps the same proportion in each split. The split is saved so you can reproduce it.
**The test set is touched exactly once, at the very end.**

In [ ]:
from sklearn.model_selection import train_test_split

train_p, temp_p, train_y, temp_y = train_test_split(paths, labels, test_size=0.30, stratify=labels, random_state=SEED)
val_p, test_p, val_y, test_y     = train_test_split(temp_p, temp_y, test_size=0.50, stratify=temp_y, random_state=SEED)

assert not (set(train_p) & set(val_p) or set(train_p) & set(test_p) or set(val_p) & set(test_p)), 'splits overlap!'
print(f'train {len(train_p)} | val {len(val_p)} | test {len(test_p)}')

with open(f'{OUT_DIR}/splits.json', 'w') as fh:
    json.dump({'class_names': class_names, 'train': train_p, 'val': val_p, 'test': test_p}, fh)

## 4. Input pipeline

Images stay as **raw RGB 0–255 floats**, resized to 224×224, the same thing `main.py` will send.
No ÷255 here: EfficientNet does its own scaling inside the model.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))   # float32, 0-255
    return img, label

def make_ds(p, y, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((p, y)).map(load, num_parallel_calls=AUTOTUNE).cache()
    if shuffle:
        ds = ds.shuffle(len(p), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

train_ds      = make_ds(train_p, train_y, shuffle=True)
train_eval_ds = make_ds(train_p, train_y)            # same images, no shuffle: for a fair train score
val_ds        = make_ds(val_p, val_y)
test_ds       = make_ds(test_p, test_y)

imgs, ys = next(iter(train_ds))
plt.figure(figsize=(12, 6))
for i in range(min(12, len(ys))):
    plt.subplot(3, 4, i + 1); plt.imshow(imgs[i].numpy().astype('uint8'))
    plt.title(class_names[int(ys[i])], fontsize=9); plt.axis('off')
plt.tight_layout(); plt.show()

## 5. Handle class imbalance

If one breed has 3× more photos, the model learns to guess it. Class weights make every breed count equally.

In [ ]:
train_counts = np.bincount(train_y, minlength=len(class_names))
class_weight = {i: len(train_y) / (len(class_names) * c) for i, c in enumerate(train_counts)}
for name, w in zip(class_names, class_weight.values()):
    print(f'{name:>14}: weight {w:.2f}')

## 6. Model: EfficientNetB0 + light head

Anti-overfitting measures built in:
- **Pre-trained features**: the base already knows edges, textures, shapes from 1.2M ImageNet photos
- **Augmentation inside the model**: active only during training, automatically off for predictions
- **Dropout + L2** on the small classification head
- **Frozen BatchNorm** during fine-tuning (unfreezing it on a small dataset is a classic way to overfit)

In [ ]:
augment = keras.Sequential([
    layers.RandomFlip('horizontal'),          # left/right only: cows are never upside down
    layers.RandomRotation(0.06),              # about ±20°: phones aren't held perfectly level
    layers.RandomZoom((-0.1, 0.2)),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.15),            # midday sun vs evening light
], name='augment')

base = keras.applications.EfficientNetB0(include_top=False, weights='imagenet', input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = False

inputs  = keras.Input((IMG_SIZE, IMG_SIZE, 3), name='image')   # raw RGB 0-255
x       = augment(inputs)
x       = base(x, training=False)            # BatchNorm stays in inference mode, also while fine-tuning
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.Dropout(0.35)(x)
outputs = layers.Dense(len(class_names), activation='softmax',
                       kernel_regularizer=keras.regularizers.l2(1e-4), name='breed')(x)
model = keras.Model(inputs, outputs, name='cow_breed_v2')
model.summary(show_trainable=True)

## 7. Phase 1: train only the new head (base frozen)

In [ ]:
# Stop when validation loss stops improving, lower the learning rate on plateaus, keep the best weights
def callbacks(tag):
    return [
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-7, verbose=1),
        keras.callbacks.ModelCheckpoint(f'/content/best_{tag}.keras', monitor='val_loss', save_best_only=True),
    ]

model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
h1 = model.fit(train_ds, validation_data=val_ds, epochs=30,
               class_weight=class_weight, callbacks=callbacks('head'))

## 8. Phase 2: fine-tune the top of the base, very gently

Unfreeze only the last ~40 layers, keep BatchNorm frozen, and use a 100× smaller learning rate.
If validation loss doesn't improve, early stopping rolls back to the phase-1 weights.

In [ ]:
base.trainable = True
for layer in base.layers[:-40]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
h2 = model.fit(train_ds, validation_data=val_ds, epochs=20,
               class_weight=class_weight, callbacks=callbacks('finetune'))

## 9. Overfitting check

**Learning curves:** training and validation loss should fall together. If validation loss turns upward
while training loss keeps falling, that's overfitting (early stopping should have caught it).

Note: training accuracy *during* `fit` is measured with augmentation + dropout switched on, so it looks lower
than it really is. The table below re-scores all three splits the same, fair way.

In [ ]:
hist = {k: h1.history[k] + h2.history[k] for k in ('loss', 'val_loss', 'accuracy', 'val_accuracy')}
split = len(h1.history['loss'])

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for a, metric, title in ((ax[0], 'loss', 'Loss'), (ax[1], 'accuracy', 'Accuracy')):
    a.plot(hist[metric], label='train', color='#1b5e37')
    a.plot(hist['val_' + metric], label='validation', color='#e0a526')
    a.axvline(split - 0.5, ls='--', color='grey'); a.text(split, a.get_ylim()[1], ' fine-tune', va='top', color='grey')
    a.set_title(title); a.set_xlabel('epoch'); a.legend()
plt.tight_layout(); plt.show()

In [ ]:
train_loss, train_acc = model.evaluate(train_eval_ds, verbose=0)
val_loss, val_acc     = model.evaluate(val_ds, verbose=0)

print(f'{"":>12}{"loss":>8}{"accuracy":>11}')
print(f'{"train":>12}{train_loss:>8.3f}{train_acc:>11.1%}')
print(f'{"validation":>12}{val_loss:>8.3f}{val_acc:>11.1%}')

gap = train_acc - val_acc
print(f'\nTrain-validation accuracy gap: {gap:.1%}')
if gap < 0.05:
    print('OK: generalises well, no meaningful overfitting.')
elif gap < 0.10:
    print('MILD overfitting: acceptable; more photos per breed would close the gap.')
else:
    print('OVERFITTING: the model memorises training photos. Fixes, in order: more/more varied photos, '
          'stronger augmentation, higher dropout (0.5), skip phase 2 or unfreeze fewer layers.')

## 10. Final test (run once)

This is the honest score: photos the model never saw and that never influenced any training decision.
Don't tweak the model based on this and re-run; that turns the test set into a second validation set.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

test_loss, test_acc = model.evaluate(test_ds, verbose=0)
probs  = model.predict(test_ds, verbose=0)
y_pred = probs.argmax(axis=1)
y_true = np.array(test_y)
print(f'TEST accuracy: {test_acc:.1%}  (validation was {val_acc:.1%}; these should be close)\n')
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

cm = confusion_matrix(y_true, y_pred, normalize='true')
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap='Greens', values_format='.0%', colorbar=False)
plt.xticks(rotation=35, ha='right'); plt.title('Test confusion matrix (row = true breed)'); plt.tight_layout(); plt.show()

## 11. Pick the "not a cow / not sure" threshold

`main.py` answers *Not a cow* when the top confidence is below a threshold (currently 48%).
This table shows the trade-off on the test set: a higher threshold makes the answers it does give more accurate,
but answers fewer photos.

In [ ]:
conf = probs.max(axis=1)
correct = y_pred == y_true
print(f'{"threshold":>10}{"answered":>10}{"accuracy when answered":>25}')
for t in (0.40, 0.48, 0.55, 0.60, 0.70, 0.80, 0.90):
    keep = conf >= t
    acc = correct[keep].mean() if keep.any() else float('nan')
    print(f'{t:>10.0%}{keep.mean():>10.0%}{acc:>25.1%}')

SUGGESTED_THRESHOLD = 0.60   # adjust after reading the table above

## 12. Save in the new `.keras` format + metadata

`model_meta.json` records the class order and versions. **Pin the same TensorFlow version in the
`my-cow-py` Dockerfile**, or loading may fail.

In [ ]:
MODEL_PATH = f'{OUT_DIR}/cow_breed_v2.keras'
model.save(MODEL_PATH)

meta = {
    'class_names': class_names,
    'img_size': IMG_SIZE,
    'input': 'RGB, float 0-255, resized to img_size (no /255 scaling)',
    'threshold': SUGGESTED_THRESHOLD,
    'tensorflow': tf.__version__,
    'keras': keras.__version__,
    'metrics': {'train_acc': round(float(train_acc), 4), 'val_acc': round(float(val_acc), 4),
                'test_acc': round(float(test_acc), 4)},
    'counts': {'train': len(train_p), 'val': len(val_p), 'test': len(test_p)},
}
with open(f'{OUT_DIR}/model_meta.json', 'w') as fh:
    json.dump(meta, fh, indent=2)

print(json.dumps(meta, indent=2))
print('\nSaved to', OUT_DIR, f'({os.path.getsize(MODEL_PATH) / 1e6:.1f} MB)')

## 13. Sanity check: reload and predict exactly like the API does

Loads the saved file fresh and runs the same steps as `main.py` (PIL → RGB → `cv2.resize` → predict).
If this disagrees with the in-notebook model, the API would give different answers than this notebook.

In [ ]:
import cv2
from PIL import Image

reloaded = keras.models.load_model(MODEL_PATH)

def predict_like_api(path):
    image = np.array(Image.open(path).convert('RGB'))
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
    p = reloaded.predict(np.expand_dims(image, 0).astype('float32'), verbose=0)[0]
    return int(p.argmax()), float(p.max())

agree = 0
sample = list(zip(test_p, test_y))[:20]
for path, true in sample:
    pred, c = predict_like_api(path)
    agree += pred == y_pred[test_p.index(path)]
print(f'API-style pipeline agrees with notebook on {agree}/{len(sample)} test images')

path, true = sample[0]
pred, c = predict_like_api(path)
print(f'Example: true={class_names[true]}  predicted={class_names[pred]} ({c:.1%})')

## 14. Download

Files are already in Google Drive at `My Drive/cow_model/`. To also download them directly:

In [ ]:
from google.colab import files
files.download(MODEL_PATH)
files.download(f'{OUT_DIR}/model_meta.json')

### Next steps
1. Copy `cow_breed_v2.keras` and `model_meta.json` into `my-cow-py/`.
2. Update `main.py` to load the `.keras` file, use 224×224, `.convert('RGB')`, read class names + threshold from `model_meta.json`.
3. Pin `tensorflow==<version printed in section 1>` in the Dockerfile.
4. To detect non-cows properly, add a `not_cow` folder (dogs, goats, sheep, empty fields, people) and re-run: the notebook picks up the new class automatically.